**`ingest_admin`**

Script examples to import administrative subdivisions for new countries

In [ ]:
from openplaces.api import get_admin
from openplaces.io.ingester import Ingester
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Recipes to import for each administrative level
ADMIN_RECIPE_IDS = {
    2: 'US_admin-nhgis-2020_admin2',  # US states
    3: 'US_admin-nhgis-2020_admin3',  # US counties
    4: 'US_admin-nhgis-2020_admin4',  # US county subdivisions
}

In [ ]:
# Reprocess downloaded data if output files exist?
REPROCESS = False

# Redownload (and reprocess) data if downloaded files exist?
REDOWNLOAD = False

# Push updates (e.g. new administrative subdivisions) to spine?
UPDATE_ADMIN_SPINE = False

# Ingest data

## ``admin2``: states / departments

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[2]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[2], verbose=True)
ingester.ingest(reprocess=REPROCESS)

In [ ]:
admin2 = get_admin(level=2, recipe=ADMIN_RECIPE_IDS[2])
admin2.sample(5).sort_index()

## ``admin3``: counties / municipalities

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[3]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[3], verbose=True)
ingester.ingest(reprocess=REPROCESS)

In [ ]:
admin3 = get_admin(level=3, recipe=ADMIN_RECIPE_IDS[3])
admin3.sample(5).sort_index()

### Add Admin IDs from recipe to reference list
Currently contains code specific to US (type, issue with cities)

In [ ]:
if UPDATE_ADMIN_SPINE:
    # Add admin IDs from new recipe to reference list of Admin IDs (spine)
    import pandas as pd
    from openplaces.api import get_admin
    from openplaces.path import recipe_path

    REGEX_ADMIN_TYPE_EXTRACT = '(Census Area|Borough|City|Municipality|Municipio)$'

    LEVEL = 3

    TEST = False

    admin_recipe = ADMIN_RECIPE_IDS[LEVEL]

    # Load admin spine
    admin_spine = get_admin(level=LEVEL, all_columns=True)
    # Load admin recipe
    admin_local = get_admin(level=LEVEL, recipe=admin_recipe)
    new_admin_ids = sorted(set(admin_local.index) - set(admin_spine.index))

    if new_admin_ids:
        print('Adding: ' + ', '.join(new_admin_ids))

        new_admin_entries = admin_local.loc[new_admin_ids].copy()

        # US-specific: extract 'Census Area', 'Borough', 'City', 'Municipality'
        new_admin_entries['type'] = (
            new_admin_entries['name_long']
            .str.title()
            .str.extract(REGEX_ADMIN_TYPE_EXTRACT)
        )
        new_admin_entries[f'admin{LEVEL}_id_source'] = 'recipe'

        # Add 'city' suffix to names of US cities sharing name with county
        new_admin_entries.loc[new_admin_entries['type'].eq('City'), 'name'] += ' city'
        new_admin_entries = new_admin_entries[
            [v for v in new_admin_entries if v in admin_spine]
        ]
        new_admin_spine = pd.concat([admin_spine, new_admin_entries]).sort_index()

        admin_recipe_path = recipe_path(
            None,
            'admin-openplaces-2026',
            filename=f'admin{LEVEL}' + ('_test' if TEST else '') + '.csv',
        )

        new_admin_spine.to_csv(admin_recipe_path, encoding='utf-8-sig')

## ``admin4``: towns / county subdivisions

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[4]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[4], verbose=True)
ingester.ingest(reprocess=REPROCESS)

In [ ]:
# US has no Admin4 in GADM spine
admin4 = get_admin('US', level=4, recipe=ADMIN_RECIPE_IDS[4], all_columns=True)
admin4

In [ ]:
if UPDATE_ADMIN_SPINE:
    # Add admin IDs from new recipe to reference list of Admin IDs (spine)
    import pandas as pd
    from openplaces.api import get_admin
    from openplaces.path import recipe_path

    REGEX_ADMIN_TYPE_EXTRACT = (
        '([Cc]ensus [Aa]rea|Borough|City|Municipality|Municipio|CCD)$'
    )

    LEVEL = 4

    TEST = False

    admin_recipe = ADMIN_RECIPE_IDS[LEVEL]

    # Load admin spine
    admin_spine = get_admin(level=LEVEL, all_columns=True, silent=True)
    # Load admin recipe
    admin_local = get_admin(level=LEVEL, recipe=admin_recipe, all_columns=True)
    new_admin_ids = sorted(set(admin_local.index) - set(admin_spine.index))

    if new_admin_ids:
        print(
            'Adding: \n'
            + '\n- '.join(new_admin_ids[:5])
            + ('\n- ...' if len(new_admin_ids) > 5 else '')
        )

        new_admin_entries = admin_local.loc[new_admin_ids].copy()

        # US-specific: extract 'Census Area', 'Borough', 'City', 'Municipality'
        new_admin_entries['type'] = (
            new_admin_entries['name_long']
            .str.title()
            .str.extract(REGEX_ADMIN_TYPE_EXTRACT)
        )
        # new_admin_entries[f'admin{LEVEL}_id_source'] = admin_recipe

        # Add 'city' suffix to names of US cities sharing name with county
        # new_admin_entries.loc[new_admin_entries['type'].eq('City'), 'name'] += ' city'
        new_admin_entries = new_admin_entries[
            [v for v in new_admin_entries if v in admin_spine]
        ]
        new_admin_spine = pd.concat([admin_spine, new_admin_entries]).sort_index()

        admin_recipe_path = recipe_path(
            None,
            'admin-openplaces-2026',
            filename=f'admin{LEVEL}' + ('_test' if TEST else '') + '.csv',
        )

        new_admin_spine.to_csv(admin_recipe_path, encoding='utf-8-sig')